#Concept 1 — ROC Curve kya hota hai?
###Accuracy batati hai kitne predictions correct hue, lekin ROC curve batata hai ki model positive aur negative classes ko different thresholds par kitna effectively distinguish kar raha hai.

##ROC graph mein:
- TPR (True Positive Rate) → actual positives mein kitne correctly detect hue.
- FPR (False Positive Rate) → actual negatives mein kitne galti se positive predict hue.

#Concept 2 — AUC (Area Under the Curve)
AUC ROC curve ke neeche ka area represent karta hai.
- 1.0 → perfect separation
- 0.5 → random-like discrimination
- 0.5 se kam → predictions ki ranking reversed discrimination dikha sakti hai
Generally, higher AUC means better class separation.

#Cell 1 — Calculate ROC-AUC for 6 Models

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Naive Bayes": GaussianNB()
}

# 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="roc_auc"
    )

    results.append({
        "Model": name,
        "Fold AUC": scores.round(4),
        "Mean AUC": scores.mean(),
        "Std AUC": scores.std()
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Mean AUC",
    ascending=False
).reset_index(drop=True)

print(results_df.round(4).to_string(index=False))

              Model                                 Fold AUC  Mean AUC  Std AUC
Logistic Regression    [0.9951, 0.9977, 0.9851, 1.0, 0.9987]    0.9953   0.0053
                SVM     [0.9997, 0.9898, 0.9851, 1.0, 0.998]    0.9945   0.0060
      Random Forest [0.9984, 0.9785, 0.9788, 0.9945, 0.9925]    0.9885   0.0083
        Naive Bayes [0.9918, 0.9761, 0.9888, 0.9848, 0.9886]    0.9860   0.0054
                KNN [0.9993, 0.9733, 0.9676, 0.9993, 0.9847]    0.9849   0.0130
      Decision Tree   [0.9253, 0.8439, 0.87, 0.9415, 0.9193]    0.9000   0.0368
